In [2]:
!pip install -q transformers datasets accelerate peft

In [3]:
import pandas as pd

df = pd.read_csv("trainingDatasetAUTLaw.csv", sep=";", encoding="utf-8-sig")

df.columns = df.columns.str.strip()
df = df[["question", "answer"]]
df = df.dropna()

# Prompt-Format für Modell
df["text"] = "### Frage:\n" + df["question"] + "\n\n### Antwort:\n" + df["answer"]

df.to_csv("train_tinyllama.csv", index=False)

print(df.head())

                                            question  \
0  Wann muss ich eine Arbeitnehmerveranlagung mac...   
1  Welche Kosten kann ich als Werbungskosten abse...   
2             Wie funktioniert die Pendlerpauschale?   
3              Was ist die Kleinunternehmerregelung?   
4       Wie hoch ist die Umsatzsteuer in Österreich?   

                                              answer  \
0  Eine Arbeitnehmerveranlagung ist verpflichtend...   
1  Werbungskosten sind beruflich veranlasste Ausg...   
2  Die Pendlerpauschale kann beantragt werden, we...   
3  Unternehmer mit einem Jahresumsatz von bis zu ...   
4  Der Normalsteuersatz beträgt 20 Prozent. Für b...   

                                                text  
0  ### Frage:\nWann muss ich eine Arbeitnehmerver...  
1  ### Frage:\nWelche Kosten kann ich als Werbung...  
2  ### Frage:\nWie funktioniert die Pendlerpausch...  
3  ### Frage:\nWas ist die Kleinunternehmerregelu...  
4  ### Frage:\nWie hoch ist die Umsatzsteuer in Ö..

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [6]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "train_tinyllama.csv"})

Generating train split: 0 examples [00:00, ? examples/s]

In [15]:
def tokenize(example):
    prompt = f"### Frage:\n{example['question']}\n\n### Antwort:\n"
    answer = example["answer"]

    full_text = prompt + answer
    tokenized = tokenizer(full_text, truncation=True, max_length=512)

    prompt_tokens = tokenizer(prompt, truncation=True, max_length=512)["input_ids"]
    labels = tokenized["input_ids"].copy()
    labels[:len(prompt_tokens)] = [-100] * len(prompt_tokens)

    tokenized["labels"] = labels
    return tokenized

dataset = dataset.map(tokenize)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

In [18]:
from transformers import TrainingArguments, Trainer
import torch

torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./tinyllama_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    fp16=True,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,1.849057
20,1.840287
30,1.941266
40,1.757065
50,1.788857
60,1.907711
70,1.782311
80,1.702923
90,1.751771
100,1.742659


TrainOutput(global_step=339, training_loss=1.631303486219198, metrics={'train_runtime': 182.1746, 'train_samples_per_second': 7.41, 'train_steps_per_second': 1.861, 'total_flos': 584802437271552.0, 'train_loss': 1.631303486219198, 'epoch': 3.0})

In [19]:
trainer.save_model("tinyllama_model")
tokenizer.save_pretrained("tinyllama_model")

('tinyllama_model/tokenizer_config.json',
 'tinyllama_model/chat_template.jinja',
 'tinyllama_model/tokenizer.json')

In [20]:
def ask(q):
    prompt = f"""### Frage:
{q}

### Antwort:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("### Antwort:")[-1].strip()

print(ask("Was ist die Einkommensteuer?"))

Die Einkommensteuer ist eine Steuer auf Einkommen. Sie wird von der Arbeitgeberbetriebskasse abgezogen. Die Höhe hängt von der Einkommensgrenze ab. Die Höhe kann bis zu 90 Prozent des Einkommens betragen. Die Steuer kann auch von der Arbeitgeberbetriebskasse abgezogen werden
